# Carga de Datos

## Contexto

En un entorno productivo, los datos provendrían directamente del Data Warehouse (DWH) o Data Lake de la empresa, resultantes de procesos ETL que materializan la información en tablas estructuradas.

Para este proyecto académico, utilizaremos un dataset de ejemplo no productivo en formato CSV (`Base_de_datos.csv`) que simula el flujo de datos que normalmente vendría de estas fuentes empresariales.


In [8]:
# Importar librerías necesarias
import json
import os
from pathlib import Path

try:
    import pandas as pd
except ImportError as exc:
    raise ImportError(
        "La librería 'pandas' no está instalada en el entorno actual. "
        "Ejecuta `pip install pandas` dentro del entorno virtual -venv antes de continuar."
    ) from exc

print("Librerías importadas correctamente")

Librerías importadas correctamente


## 1. Preparación del entorno
En esta sección validamos la correcta importación de librerías y dejamos registrado el contexto en el que se ejecuta el notebook. Este paso garantiza reproducibilidad del pipeline, especialmente cuando se ejecuta desde jobs automatizados.


In [9]:
# Cargar configuración del proyecto
config_path = Path("../../config.json")

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

project_identifier = config.get("project_code", "desconocido")
pipeline_config = config.get("pipeline_config", {})

if "data_path" not in pipeline_config:
    raise KeyError(
        "El archivo config.json no contiene la clave 'pipeline_config[\"data_path\"]'. "
        "Verifica la configuración del proyecto."
    )

paths_config = pipeline_config

data_full_path = Path("../../") / paths_config["data_path"]

print(f"Proyecto: {project_identifier}")
print(f"Ruta del dataset: {data_full_path}")
print(f"¿Existe el archivo? {data_full_path.exists()}")
print(f"Rutas configuradas: {paths_config}")

Proyecto: proyecto_ml
Ruta del dataset: ..\..\Base_de_datos.csv
¿Existe el archivo? True
Rutas configuradas: {'data_path': 'Base_de_datos.csv', 'model_path': 'mlops_pipeline/models/', 'output_path': 'mlops_pipeline/outputs/', 'log_path': 'mlops_pipeline/logs/'}


### Validación de la fuente de datos
La ruta se obtiene desde la configuración centralizada. Validamos su existencia antes de continuar para evitar fallos posteriores en la cadena MLOps.


In [10]:
# Cargar el dataset desde CSV
# Nota: El CSV usa punto y coma (;) como separador
df = pd.read_csv(data_full_path, sep=';', encoding='utf-8')

print(f"Dataset cargado exitosamente")
print(f"Dimensiones del dataset: {df.shape}")
print(f"Columnas: {df.shape[1]}")
print(f"Filas: {df.shape[0]}")

Dataset cargado exitosamente
Dimensiones del dataset: (4424, 37)
Columnas: 37
Filas: 4424


### Lectura del dataset
Se utiliza `pandas.read_csv` con separador `;` ya que el archivo proviene de un proceso ETL europeo. Esta lectura es el punto de entrada oficial del pipeline de datos.


In [11]:
# Visualización inicial del dataset
print("=" * 60)
print("PRIMERAS FILAS DEL DATASET")
print("=" * 60)
df.head()

PRIMERAS FILAS DEL DATASET


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [12]:
# Información general del dataset
print("=" * 60)
print("INFORMACIÓN DEL DATASET")
print("=" * 60)
df.info()

INFORMACIÓN DEL DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance	                     4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualific

In [13]:
# Verificar valores faltantes
print("=" * 60)
print("VALORES FALTANTES POR COLUMNA")
print("=" * 60)
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Valores Faltantes': missing_values,
    'Porcentaje': missing_percent
})
missing_df = missing_df[missing_df['Valores Faltantes'] > 0].sort_values('Valores Faltantes', ascending=False)

if len(missing_df) > 0:
    print(missing_df)
else:
    print("No hay valores faltantes en el dataset")

VALORES FALTANTES POR COLUMNA
No hay valores faltantes en el dataset


### Control de calidad inicial
Antes de continuar con el EDA verificamos la presencia de valores faltantes y el balance de la variable objetivo. Estos indicadores alimentar�n las reglas de validaci�n autom�ticas en etapas posteriores.


### 3. Perfilado estructural del dataset
Identificamos los tipos de variables y caracter�sticas relevantes para que el equipo de ingenier�a valide expectativas de esquema (contract tests) y mantenga controlados los cambios en la fuente.


In [14]:
from pandas.api.types import is_numeric_dtype, is_bool_dtype

def infer_feature_type(series: pd.Series) -> str:
    if is_bool_dtype(series):
        return "booleana"
    if is_numeric_dtype(series):
        return "numérica"
    return "categórica"

schema_summary = pd.DataFrame({
    "columna": df.columns,
    "dtype_pandas": df.dtypes.astype(str),
    "nulos": df.isnull().sum(),
    "porcentaje_nulos": (df.isnull().sum() / len(df)).round(4) * 100,
    "valores_únicos": df.nunique()
})

schema_summary["tipo_inferido"] = [infer_feature_type(df[col]) for col in df.columns]

schema_summary.sort_values("columna", inplace=True)

schema_summary


,columna,dtype_pandas,nulos,porcentaje_nulos,valores_únicos,tipo_inferido
Admission grade,Admission grade,float64,0,0.0,620,numérica
Age at enrollment,Age at enrollment,int64,0,0.0,46,numérica
Application mode,Application mode,int64,0,0.0,18,numérica
Application order,Application order,int64,0,0.0,8,numérica
Course,Course,int64,0,0.0,17,numérica
Curricular units 1st sem (approved),Curricular units 1st sem (approved),int64,0,0.0,23,numérica
Curricular units 1st sem (credited),Curricular units 1st sem (credited),int64,0,0.0,21,numérica
Curricular units 1st sem (enrolled),Curricular units 1st sem (enrolled),int64,0,0.0,23,numérica
Curricular units 1st sem (evaluations),Curricular units 1st sem (evaluations),int64,0,0.0,35,numérica
Curricular units 1st sem (grade),Curricular units 1st sem (grade),float64,0,0.0,805,numérica


### 4. Estad�sticos b�sicos por tipo de variable
En esta subsecci�n registramos indicadores clave (tendencia central y dispersi�n) que los equipos de QA pueden convertir en reglas autom�ticas de validaci�n.


In [15]:
numeric_cols = schema_summary.loc[schema_summary["tipo_inferido"] == "numérica", "columna"].tolist()
categorical_cols = schema_summary.loc[schema_summary["tipo_inferido"] == "categórica", "columna"].tolist()

print("Estadísticos para variables numéricas")
describe_numeric = df[numeric_cols].describe().T
describe_numeric[["mean", "std", "min", "25%", "50%", "75%", "max"]].round(2)



Estadísticos para variables numéricas


,mean,std,min,25%,50%,75%,max
Admission grade,126.98,14.48,95.00,117.90,126.10,134.80,190.00
Age at enrollment,23.27,7.59,17.00,19.00,20.00,25.00,70.00
Application mode,18.67,17.48,1.00,1.00,17.00,39.00,57.00
Application order,1.73,1.31,0.00,1.00,1.00,2.00,9.00
Course,8856.64,2063.57,33.00,9085.00,9238.00,9556.00,9991.00
Curricular units 1st sem (approved),4.71,3.09,0.00,3.00,5.00,6.00,26.00
Curricular units 1st sem (credited),0.71,2.36,0.00,0.00,0.00,0.00,20.00
Curricular units 1st sem (enrolled),6.27,2.48,0.00,5.00,6.00,7.00,26.00
Curricular units 1st sem (evaluations),8.30,4.18,0.00,6.00,8.00,10.00,45.00
Curricular units 1st sem (grade),10.64,4.84,0.00,11.00,12.29,13.40,18.88


In [16]:
print("\nFrecuencias para variables categóricas (top 5 valores)")
categorical_summary = {}
for col in categorical_cols:
    categorical_summary[col] = df[col].value_counts().head().to_dict()

pd.DataFrame.from_dict(categorical_summary, orient="index")



Frecuencias para variables categóricas (top 5 valores)


,Graduate,Dropout,Enrolled
Target,2209,1421,794


### 5. Reglas de integridad m�nimas
Validamos duplicados y rangos b�sicos para detectar anomal�as tempranas antes del entrenamiento de modelos.


In [17]:
integrity_checks = {
    "duplicados_totales": int(df.duplicated().sum()),
    "filas_con_valores_negativos": int((df[numeric_cols] < 0).any(axis=1).sum()),
}

integrity_checks


{'duplicados_totales': 0, 'filas_con_valores_negativos': 2101}

In [18]:
# Verificar la variable objetivo (Target)
print("=" * 60)
print("DISTRIBUCIÓN DE LA VARIABLE OBJETIVO")
print("=" * 60)
if 'Target' in df.columns:
    target_dist = df['Target'].value_counts()
    target_percent = df['Target'].value_counts(normalize=True) * 100
    
    target_info = pd.DataFrame({
        'Frecuencia': target_dist,
        'Porcentaje': target_percent
    })
    print(target_info)
    print(f"\nTotal de registros: {len(df)}")
else:
    print("No se encontró la columna 'Target' en el dataset")


DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
          Frecuencia  Porcentaje
Target                          
Graduate        2209   49.932188
Dropout         1421   32.120253
Enrolled         794   17.947559

Total de registros: 4424


### 6. Registro de metadatos operativos
Para integrar este paso al pipeline MLOps, persistimos un resumen con informaci�n clave del dataset (forma, columnas, fecha de extracci�n). Este artefacto servir� para auditor�as y validaciones automatizadas en etapas posteriores.


In [19]:
# Persistir metadatos del dataset para trazabilidad
from datetime import datetime

output_path = paths_config.get("output_path")
if output_path is None:
    raise KeyError(
        "El archivo config.json no contiene la clave 'project_code[\"output_path\"]'. "
        "Actualiza la configuración antes de generar metadatos."
    )

output_dir = Path("../../") / output_path
output_dir.mkdir(parents=True, exist_ok=True)

metadata = {
    "dataset_name": "student_dropout_retention",
    "source_file": str(data_full_path.resolve()),
    "extraction_timestamp": datetime.utcnow().isoformat() + "Z",
    "row_count": int(df.shape[0]),
    "column_count": int(df.shape[1]),
    "columns": schema_summary.to_dict(orient="records"),
}

metadata_path = output_dir / "raw_data_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as meta_file:
    json.dump(metadata, meta_file, indent=2, ensure_ascii=False)

print(f"Metadatos guardados en: {metadata_path}")
metadata_path


Metadatos guardados en: ..\..\mlops_pipeline\outputs\raw_data_metadata.json


C:\Users\jesus\AppData\Local\Temp\ipykernel_230988\2192812917.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "extraction_timestamp": datetime.utcnow().isoformat() + "Z",


WindowsPath('../../mlops_pipeline/outputs/raw_data_metadata.json')

## 7. Resumen ejecutivo
- Fuente validada mediante `config.json`; el archivo `Base_de_datos.csv` est� disponible y sin incidencias de acceso.
- No se identificaron valores faltantes ni registros duplicados en la extracci�n inicial.
- El conjunto contiene columnas mayoritariamente num�ricas que modelan el desempe�o acad�mico, junto con variables categ�ricas administrativas.
- La variable objetivo `Target` presenta tres clases (`Dropout`, `Enrolled`, `Graduate`) con distribuci�n desequilibrada que deber� considerarse en la fase de modelado.

El dataframe `df` queda disponible para las siguientes etapas del pipeline, especialmente para el notebook de **An�lisis Exploratorio de Datos (EDA)**.

### 8. Espacio para comentarios del analista
> **Instrucciones:** Documenta aqu� tus observaciones, hip�tesis o dudas respecto a la calidad y procedencia de los datos. Esta secci�n alimentar� el backlog de mejoras y deber� revisarse en las reuniones de seguimiento.

- Comentario 1:
- Comentario 2:
- Acciones sugeridas:
